In [45]:
import random

In [ ]:
class Stat:
    name:str
    amount:float

In [ ]:
class Buff:
    name:str
    stat:str
    mod:float

In [85]:
class Attack:
    name:str
    dmg:int
    VCD:float
    FCD:float
    CD:float
    Animation:float
    crit_chance = 1
    proc_chance:float=0
    proc:super

    def cast(self):
        self.CD = self.FCD + self.VCD
        return self.dmg

    def update(self, tick):
        self.CD = max(self.CD - tick, 0)

    def crit_tick(self):
        pass

    def is_crit(self):
        return random.random() <= self.crit_chance

    def is_proc(self):
        return random.random() <= self.proc_chance

    def __repr__(self):
        return self.name

class TriangleShot(Attack):
    name='Triangle Shot'
    dmg=312*3
    FCD=6
    VCD=35
    CD=0
    Animation = 0.6

    def crit_tick(self):
        self.update(1)

class DoubleStrafe(Attack):
    name='Double Strafe'
    dmg=286*2*1.55
    FCD=4
    VCD=35
    CD=0
    Animation = 0.6

    def crit_tick(self):
        self.update(1)

class RepeatingStrafe(Attack):
    name='Repeating Strafe'
    dmg=286*2
    CD=0
    Animation = 0.6

class NormalAttack(Attack):
    name='Normal Attack'
    dmg=100
    FCD=0
    VCD=0
    CD=0
    Animation = 1/6
    proc = RepeatingStrafe
    proc_chance = 0.08

In [86]:
class AttackOrder:
    queue = []

    def add_attack(self, attack:Attack):
        self.queue.append(attack)
        return self

    @classmethod
    def tick_skills(cls, tick):
        [skill.update(tick) for skill in cls.queue]

    @classmethod
    def crit_tick_skills(cls):
        [skill.crit_tick() for skill in cls.queue]

    def __repr__(self):
        return str(self.queue)

In [87]:
pipe = AttackOrder()
pipe.add_attack(TriangleShot()).add_attack(DoubleStrafe()).add_attack(NormalAttack())


[Triangle Shot, Double Strafe, Normal Attack]

In [88]:
time = 0
max_time = 300

attack_order=[]

while time < max_time:
    for skill in pipe.queue:
        if skill.CD == 0:
            skill.cast()
            attack_order.append(skill)
            if isinstance(skill, NormalAttack) and skill.is_crit():
                pipe.crit_tick_skills()

            if isinstance(skill, NormalAttack) and skill.is_proc():
                attack_order.append(skill.proc)

            time += skill.Animation
            pipe.tick_skills(skill.Animation)
            break


In [90]:
total_dmg = sum([attack.dmg for attack in attack_order])

print(f'No Haste: {total_dmg/300}')

No Haste: 1352.9756666666667


In [84]:
total_dmg = sum([attack.dmg for attack in attack_order])
total_dmg/300

1317.28